In [1]:
# All package imports (run this cell first)
import sys
import subprocess
import json
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`.

In [29]:
# Kernel verification
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")
print(f"Using root .venv: {'✓ Yes' if _venv_ok else '✗ No – select Kernel → Python (My-Crew-Manager .venv)'}")

Python: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\.venv\Scripts\python.exe
Using root .venv: ✓ Yes


# Model Part 1 - Proposal Summarization and Sprint Planning

From raw project description to structured Part 1 output (summary, roles, features, goals, timeline).

Run from AI folder or project root. This notebook expects an existing Model 1 JSONL dataset in llms/fine_tune/dataset.

## Setup paths

In [2]:
# Resolve AI root
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_model1_overview_lora"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

AI root: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI
Dataset: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\dataset
Output: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora


## Step 1: Import existing Model 1 dataset

Load the prepared `model1_description_to_part1.jsonl` dataset and preview sample records.

In [31]:
# Import and preview Model 1 dataset
dataset_file = DATASET_DIR / "model1_description_to_part1.jsonl"
if not dataset_file.exists():
    raise FileNotFoundError(
        f"Missing dataset: {dataset_file}. Generate or convert datasets first."
    )

rows = []
for line in dataset_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples from {dataset_file.name}")
print("Model 1 contract:")
print("- Prompt: Proposal / Input text from user")
print("- Response: Title + Summary + Roles + Features + Goals + Timeline")
print("- Excluded from response: Status, Proposal / Input, Backlog")
if rows:
    print("\nSample prompt:")
    print(rows[0]["prompt"][:220] + ("..." if len(rows[0]["prompt"]) > 220 else ""))
    print("\nSample response snippet:")
    print(rows[0]["response"][:280] + ("..." if len(rows[0]["response"]) > 280 else ""))

Loaded 26 examples from model1_description_to_part1.jsonl
Model 1 contract:
- Prompt: Proposal / Input text from user
- Response: Title + Summary + Roles + Features + Goals + Timeline
- Excluded from response: Status, Proposal / Input, Backlog

Sample prompt:
Develop the MyCrewManager platform, a web-based system that enables project managers to assign tasks, track progress, and receive AI-generated recommendations for team coordination and risk mitigation. The project will c...

Sample response snippet:
=== MyCrewManager ===
Summary:
Develop the MyCrewManager platform, a web-based system that enables project managers to assign tasks, track progress, and receive AI-generated recommendations for team coordination and risk mitigation. The project will consist of a Django REST API b...


## Step 2: Prepare tokenized dataset

In [3]:
from llms.fine_tune.prepare_dataset import prepare_model1_dataset

DATASET_FILENAME = "model1_description_to_part1_dualprompt.jsonl"
source_dataset_file = DATASET_DIR / DATASET_FILENAME
tokenized_path = TOKENIZED_DIR / "tokenized_model1_qwen_dualprompt"
MAX_LENGTH = 512
FORCE_REBUILD_TOKENIZED = False

if not source_dataset_file.exists():
    raise FileNotFoundError(f"Missing source dataset: {source_dataset_file}")

source_count = sum(1 for line in source_dataset_file.read_text(encoding="utf-8").splitlines() if line.strip())
print(f"Source examples in JSONL ({DATASET_FILENAME}): {source_count}")

rebuild_needed = FORCE_REBUILD_TOKENIZED or not tokenized_path.exists()
if tokenized_path.exists() and not rebuild_needed:
    dataset = load_from_disk(str(tokenized_path))
    tokenized_count = len(dataset)
    if tokenized_count != source_count:
        print(
            f"Tokenized dataset is stale (tokenized={tokenized_count}, source={source_count}). "
            f"Rebuilding..."
        )
        rebuild_needed = True

if rebuild_needed:
    if tokenized_path.exists():
        import shutil
        shutil.rmtree(tokenized_path)
    dataset = prepare_model1_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        dataset_filename=DATASET_FILENAME,
        output_dir=str(tokenized_path),
    )
    print(f"Rebuilt tokenized dataset at {tokenized_path}")
else:
    print(f"Loaded tokenized dataset from {tokenized_path}")

# 60/20/20 split: train / eval / holdout
split_primary = dataset.train_test_split(test_size=0.4, seed=42)
train_dataset = split_primary["train"]

split_secondary = split_primary["test"].train_test_split(test_size=0.5, seed=42)
eval_dataset = split_secondary["train"]
holdout_dataset = split_secondary["test"]

print(f"Total tokenized examples: {len(dataset)}")
print(f"Train size (60%): {len(train_dataset)}")
print(f"Eval size (20%): {len(eval_dataset)}")
print(f"Holdout size (20%): {len(holdout_dataset)}")
print(f"Max length: {MAX_LENGTH}")
print("\nUsing dual-prompt training dataset to improve instruction compliance.")

Source examples in JSONL (model1_description_to_part1_dualprompt.jsonl): 52


Tokenizing Model 1:   0%|          | 0/52 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/52 [00:00<?, ? examples/s]

Saved Model 1 tokenized dataset to c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\tokenized\tokenized_model1_qwen_dualprompt
Rebuilt tokenized dataset at c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\tokenized\tokenized_model1_qwen_dualprompt
Total tokenized examples: 52
Train size (60%): 31
Eval size (20%): 10
Holdout size (20%): 11
Max length: 512

Using dual-prompt training dataset to improve instruction compliance.


## Step 3: Load model & apply LoRA

In [4]:
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## Step 4: Train iteratively until target

Train in capped rounds, evaluate after each round, and stop early when quality targets are reached.

In [5]:
import gc
import inspect
import math
import os
import random
import shutil
from pathlib import Path

os.environ.setdefault("TENSORBOARD_LOGGING_DIR", str(OUTPUT_DIR / "logs"))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

TARGET_EVAL_LOSS = 1.20
TARGET_PERPLEXITY = 3.50
MAX_ROUNDS = 8
MIN_ROUNDS = 2
PATIENCE_ROUNDS = 2
EPOCHS_PER_ROUND = 1
SEEDS = [42, 123, 456]

# Memory-safe effective batch size: 2 (1 x accumulation 2)
TRAIN_BATCH_SIZE = 1
GRAD_ACC_STEPS = 2

TRIALS_DIR = OUTPUT_DIR / "trials"
TRIALS_DIR.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clear_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


training_kwargs_base = {
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACC_STEPS,
    "per_device_eval_batch_size": 1,
    "num_train_epochs": EPOCHS_PER_ROUND,
    "logging_steps": 10,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "save_strategy": "no",
    "dataloader_pin_memory": False,
    "prediction_loss_only": True,
    "torch_empty_cache_steps": 1,
    "disable_tqdm": True,
}
if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs_base["evaluation_strategy"] = "no"
else:
    training_kwargs_base["eval_strategy"] = "no"

trial_summaries = []
global_best = None

for seed in SEEDS:
    print(f"\n{'=' * 80}")
    print(f"Starting trial for seed={seed}")
    print(f"{'=' * 80}")

    set_global_seed(seed)
    clear_cuda_cache()

    trial_dir = TRIALS_DIR / f"seed_{seed}"
    if trial_dir.exists():
        shutil.rmtree(trial_dir)
    trial_dir.mkdir(parents=True, exist_ok=True)

    trial_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    if torch.cuda.is_available():
        trial_model = trial_model.to("cuda")

    trial_model = get_peft_model(trial_model, peft_config)

    trial_kwargs = dict(training_kwargs_base)
    trial_kwargs["output_dir"] = str(trial_dir)
    trial_kwargs["seed"] = seed
    trial_kwargs["data_seed"] = seed

    trial_args = TrainingArguments(**trial_kwargs)

    trainer = Trainer(
        model=trial_model,
        args=trial_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    round_history = []
    best_round = None
    best_checkpoint_dir = None
    rounds_without_improvement = 0
    stop_reason = "max_rounds_reached"

    for round_num in range(1, MAX_ROUNDS + 1):
        print(f"\n--- Seed {seed} | Round {round_num}/{MAX_ROUNDS} ---")
        train_result = trainer.train()

        pred_output = trainer.predict(eval_dataset, metric_key_prefix="eval")
        eval_metrics = pred_output.metrics
        eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")
        perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")

        round_checkpoint_dir = trial_dir / f"round_{round_num}"
        round_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(round_checkpoint_dir))

        round_entry = {
            "seed": seed,
            "round": round_num,
            "train_loss": float(train_result.training_loss),
            "eval_loss": float(eval_loss) if eval_loss is not None else None,
            "perplexity": float(perplexity),
            "checkpoint_dir": str(round_checkpoint_dir),
        }
        round_history.append(round_entry)

        print(f"Train loss: {round_entry['train_loss']:.4f}")
        if eval_loss is not None:
            print(f"Eval loss: {eval_loss:.4f}")
        else:
            print("Eval loss: unavailable")
        print(f"Perplexity: {perplexity:.4f}" if math.isfinite(perplexity) else "Perplexity: inf")

        is_better = False
        if eval_loss is not None:
            if best_round is None:
                is_better = True
            else:
                if eval_loss < best_round["eval_loss"]:
                    is_better = True
                elif eval_loss == best_round["eval_loss"] and perplexity < best_round["perplexity"]:
                    is_better = True

        if is_better:
            best_round = round_entry
            best_checkpoint_dir = round_checkpoint_dir
            rounds_without_improvement = 0
            print("Improvement detected: updated best checkpoint.")
        else:
            rounds_without_improvement += 1
            print(f"No improvement. Patience counter: {rounds_without_improvement}/{PATIENCE_ROUNDS}")

        if eval_loss is not None and (eval_loss <= TARGET_EVAL_LOSS or perplexity <= TARGET_PERPLEXITY):
            if round_num >= MIN_ROUNDS:
                stop_reason = "target_reached"
                print(
                    f"Stopping: target reached at round {round_num} "
                    f"(eval_loss={eval_loss:.4f}, perplexity={perplexity:.4f})"
                )
                break

        if round_num >= MIN_ROUNDS and rounds_without_improvement >= PATIENCE_ROUNDS:
            stop_reason = "patience_exhausted"
            print(f"Stopping: no improvement for {PATIENCE_ROUNDS} rounds.")
            break

        clear_cuda_cache()

    if best_checkpoint_dir is None:
        print(f"Seed {seed}: no valid checkpoint selected; skipping holdout evaluation.")
        del trainer, trial_model
        clear_cuda_cache()
        continue

    holdout_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    best_model = PeftModel.from_pretrained(holdout_base, str(best_checkpoint_dir))
    if torch.cuda.is_available():
        best_model = best_model.to("cuda")

    holdout_args = TrainingArguments(
        output_dir=str(trial_dir / "holdout_eval"),
        report_to="none",
        per_device_eval_batch_size=1,
        dataloader_pin_memory=False,
        prediction_loss_only=True,
        disable_tqdm=True,
    )
    holdout_trainer = Trainer(
        model=best_model,
        args=holdout_args,
        eval_dataset=holdout_dataset,
        processing_class=tokenizer,
    )

    holdout_pred = holdout_trainer.predict(holdout_dataset, metric_key_prefix="holdout")
    holdout_metrics = holdout_pred.metrics
    holdout_loss = holdout_metrics.get("holdout_loss") or holdout_metrics.get("test_loss")
    holdout_perplexity = math.exp(holdout_loss) if holdout_loss is not None and holdout_loss < 20 else float("inf")

    trial_summary = {
        "seed": seed,
        "stop_reason": stop_reason,
        "rounds_completed": len(round_history),
        "best_round": best_round,
        "best_checkpoint_dir": str(best_checkpoint_dir),
        "holdout_loss": float(holdout_loss) if holdout_loss is not None else None,
        "holdout_perplexity": float(holdout_perplexity),
    }
    trial_summaries.append(trial_summary)

    print(f"\nSeed {seed} summary")
    print(f"- Stop reason: {stop_reason}")
    print(f"- Best round: {best_round['round']} (eval_loss={best_round['eval_loss']:.4f}, perplexity={best_round['perplexity']:.4f})")
    print(
        f"- Holdout: loss={trial_summary['holdout_loss']:.4f}, "
        f"perplexity={trial_summary['holdout_perplexity']:.4f}"
        if trial_summary["holdout_loss"] is not None
        else "- Holdout: unavailable"
    )

    if global_best is None:
        global_best = trial_summary
    else:
        cur_loss = trial_summary["holdout_loss"]
        best_loss = global_best["holdout_loss"]
        cur_ppl = trial_summary["holdout_perplexity"]
        best_ppl = global_best["holdout_perplexity"]

        if cur_loss is not None and (best_loss is None or cur_loss < best_loss):
            global_best = trial_summary
        elif cur_loss is not None and best_loss is not None and cur_loss == best_loss and cur_ppl < best_ppl:
            global_best = trial_summary

    del holdout_trainer, best_model, holdout_base, trainer, trial_model
    clear_cuda_cache()

if not trial_summaries:
    raise RuntimeError("No successful trial completed. Cannot promote best model.")

print(f"\n{'=' * 80}")
print("TRIAL RESULTS")
print(f"{'=' * 80}")
for t in trial_summaries:
    br = t["best_round"]
    holdout_loss_txt = f"{t['holdout_loss']:.4f}" if t["holdout_loss"] is not None else "NA"
    print(
        f"Seed {t['seed']}: stop={t['stop_reason']}, rounds={t['rounds_completed']}, "
        f"best_round={br['round']} (eval_loss={br['eval_loss']:.4f}, perplexity={br['perplexity']:.4f}), "
        f"holdout_loss={holdout_loss_txt}, holdout_perplexity={t['holdout_perplexity']:.4f}"
    )

print(f"\nGlobal winner seed: {global_best['seed']}")
print(f"Winner checkpoint: {global_best['best_checkpoint_dir']}")

winner_seed = global_best["seed"]
winner_checkpoint_dir = Path(global_best["best_checkpoint_dir"])
winner_holdout_loss = global_best["holdout_loss"]
winner_holdout_perplexity = global_best["holdout_perplexity"]
training_trial_summaries = trial_summaries


Starting trial for seed=42


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



--- Seed 42 | Round 1/8 ---
{'loss': '1.721', 'grad_norm': '0.7829', 'learning_rate': '2.187e-05', 'epoch': '0.6452'}
{'train_runtime': '13.3', 'train_samples_per_second': '2.33', 'train_steps_per_second': '1.203', 'train_loss': '1.684', 'epoch': '1'}
Train loss: 1.6839
Eval loss: 1.6265
Perplexity: 5.0862
Improvement detected: updated best checkpoint.

--- Seed 42 | Round 2/8 ---
{'loss': '1.615', 'grad_norm': '0.775', 'learning_rate': '2.187e-05', 'epoch': '0.6452'}
{'train_runtime': '9.644', 'train_samples_per_second': '3.215', 'train_steps_per_second': '1.659', 'train_loss': '1.58', 'epoch': '1'}
Train loss: 1.5804
Eval loss: 1.5271
Perplexity: 4.6047
Improvement detected: updated best checkpoint.

--- Seed 42 | Round 3/8 ---
{'loss': '1.513', 'grad_norm': '0.8287', 'learning_rate': '2.187e-05', 'epoch': '0.6452'}
{'train_runtime': '9.825', 'train_samples_per_second': '3.155', 'train_steps_per_second': '1.628', 'train_loss': '1.478', 'epoch': '1'}
Train loss: 1.4779
Eval loss: 1.4

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



--- Seed 456 | Round 1/8 ---
{'loss': '1.716', 'grad_norm': '0.7545', 'learning_rate': '2.187e-05', 'epoch': '0.6452'}
{'train_runtime': '10.12', 'train_samples_per_second': '3.064', 'train_steps_per_second': '1.582', 'train_loss': '1.694', 'epoch': '1'}
Train loss: 1.6940
Eval loss: 1.6317
Perplexity: 5.1126
Improvement detected: updated best checkpoint.

--- Seed 456 | Round 2/8 ---
{'loss': '1.609', 'grad_norm': '0.7366', 'learning_rate': '2.187e-05', 'epoch': '0.6452'}
{'train_runtime': '10.34', 'train_samples_per_second': '2.998', 'train_steps_per_second': '1.547', 'train_loss': '1.59', 'epoch': '1'}
Train loss: 1.5898
Eval loss: 1.5335
Perplexity: 4.6343
Improvement detected: updated best checkpoint.

--- Seed 456 | Round 3/8 ---
{'loss': '1.503', 'grad_norm': '0.8048', 'learning_rate': '2.187e-05', 'epoch': '0.6452'}
{'train_runtime': '10.28', 'train_samples_per_second': '3.015', 'train_steps_per_second': '1.556', 'train_loss': '1.485', 'epoch': '1'}
Train loss: 1.4854
Eval los

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Seed 456 summary
- Stop reason: target_reached
- Best round: 5 (eval_loss=1.2102, perplexity=3.3543)
- Holdout: loss=1.2282, perplexity=3.4152

TRIAL RESULTS
Seed 42: stop=target_reached, rounds=5, best_round=5 (eval_loss=1.2044, perplexity=3.3348), holdout_loss=1.2300, holdout_perplexity=3.4213
Seed 123: stop=target_reached, rounds=5, best_round=5 (eval_loss=1.2352, perplexity=3.4392), holdout_loss=1.2532, holdout_perplexity=3.5015
Seed 456: stop=target_reached, rounds=5, best_round=5 (eval_loss=1.2102, perplexity=3.3543), holdout_loss=1.2282, holdout_perplexity=3.4152

Global winner seed: 456
Winner checkpoint: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora\trials\seed_456\round_5


## Step 4.1: Evaluate model performance

Compute evaluation loss and perplexity on the held-out evaluation split.

In [6]:
# Evaluate promoted winner checkpoint on eval and holdout splits
if "winner_checkpoint_dir" not in globals():
    raise RuntimeError("Run the training cell first to select a winner checkpoint.")

winner_checkpoint_dir = Path(winner_checkpoint_dir)
if not winner_checkpoint_dir.exists():
    raise RuntimeError(f"Winner checkpoint not found: {winner_checkpoint_dir}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
winner_model = PeftModel.from_pretrained(eval_base, str(winner_checkpoint_dir))
if torch.cuda.is_available():
    winner_model = winner_model.to("cuda")

eval_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "winner_eval"),
    report_to="none",
    per_device_eval_batch_size=1,
    dataloader_pin_memory=False,
    prediction_loss_only=True,
    disable_tqdm=True,
)
eval_trainer = Trainer(
    model=winner_model,
    args=eval_args,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

eval_metrics = eval_trainer.predict(eval_dataset, metric_key_prefix="eval").metrics
eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")

holdout_metrics = eval_trainer.predict(holdout_dataset, metric_key_prefix="holdout").metrics
holdout_loss = holdout_metrics.get("holdout_loss") or holdout_metrics.get("test_loss")

eval_perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")
holdout_perplexity = math.exp(holdout_loss) if holdout_loss is not None and holdout_loss < 20 else float("inf")

print("Winner checkpoint evaluation")
print(f"Winner seed: {winner_seed}")
print(f"Winner checkpoint: {winner_checkpoint_dir}")
print(f"Eval loss: {eval_loss:.4f}" if eval_loss is not None else "Eval loss: unavailable")
print(f"Eval perplexity: {eval_perplexity:.4f}" if math.isfinite(eval_perplexity) else "Eval perplexity: inf")
print(f"Holdout loss: {holdout_loss:.4f}" if holdout_loss is not None else "Holdout loss: unavailable")
print(f"Holdout perplexity: {holdout_perplexity:.4f}" if math.isfinite(holdout_perplexity) else "Holdout perplexity: inf")

print("\nAll eval metrics:")
print(eval_metrics)
print("\nAll holdout metrics:")
print(holdout_metrics)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Winner checkpoint evaluation
Winner seed: 456
Winner checkpoint: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora\trials\seed_456\round_5
Eval loss: 1.2101
Eval perplexity: 3.3538
Holdout loss: 1.2282
Holdout perplexity: 3.4152

All eval metrics:
{'eval_loss': 1.210101842880249, 'eval_model_preparation_time': 0.0115, 'eval_runtime': 1.2285, 'eval_samples_per_second': 8.14, 'eval_steps_per_second': 8.14}

All holdout metrics:
{'holdout_loss': 1.2282319068908691, 'holdout_model_preparation_time': 0.0115, 'holdout_runtime': 1.2816, 'holdout_samples_per_second': 8.583, 'holdout_steps_per_second': 8.583}


## Step 5: Save adapter

In [7]:
import shutil

if "winner_checkpoint_dir" not in globals():
    raise RuntimeError("Run training first so a winner checkpoint is selected.")

winner_checkpoint_dir = Path(winner_checkpoint_dir)
if not winner_checkpoint_dir.exists():
    raise RuntimeError(f"Winner checkpoint missing: {winner_checkpoint_dir}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Clean existing adapter artifacts (keep trials folder if present)
for child in OUTPUT_DIR.iterdir():
    if child.name == "trials":
        continue
    if child.is_dir():
        shutil.rmtree(child)
    else:
        child.unlink()

# Copy winner adapter into final output directory
for child in winner_checkpoint_dir.iterdir():
    dst = OUTPUT_DIR / child.name
    if child.is_dir():
        shutil.copytree(child, dst)
    else:
        shutil.copy2(child, dst)

# Ensure tokenizer files are present for inference
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f"Promoted winner checkpoint from: {winner_checkpoint_dir}")
print(f"Saved final adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:")
print("  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_model1_overview_lora")

Promoted winner checkpoint from: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora\trials\seed_456\round_5
Saved final adapter and tokenizer to c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora
Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:
  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_model1_overview_lora


## Step 6: Quick inference test

In [11]:
import re
import json

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

GUIDED_PREFIX = (
    "Generate ONLY the following sections in order: "
    "Title, Summary, Roles, Features, Goals, Timeline (Week 1-4).\n"
    "Do not include Status, Proposal/Input, or Backlog.\n\n"
    "Proposal / Input:\n"
)


def _infer_title_from_proposal(proposal: str) -> str:
    words = re.findall(r"[A-Za-z0-9]+", proposal)
    if not words:
        return "Project Overview"
    return " ".join(words[:4]).title() + " Project"


def _normalize_headings(text: str) -> str:
    normalized = text
    replacements = [
        (r"(?im)^\s*===\s*Title\s*===\s*$", "Title:"),
        (r"(?im)^\s*===\s*Summary\s*===\s*$", "Summary:"),
        (r"(?im)^\s*===\s*Roles\s*===\s*$", "Roles:"),
        (r"(?im)^\s*===\s*Features\s*===\s*$", "Features:"),
        (r"(?im)^\s*===\s*Goals\s*===\s*$", "Goals:"),
        (r"(?im)^\s*===\s*Timeline\s*===\s*$", "Timeline:"),
        (r"(?im)^\s*Project\s+Description\s*:\s*$", "Summary:"),
        (r"(?im)^\s*Timeline\s*\(Week\s*1-4\)\s*:\s*$", "Timeline:"),
        (r"(?im)^\s*Timeline\s*\(Week\s*2-4\)\s*:\s*$", "Timeline:"),
    ]
    for pattern, repl in replacements:
        normalized = re.sub(pattern, repl, normalized)

    if not re.search(r"(?im)^\s*Title\s*:", normalized):
        m = re.search(r"(?im)^\s*===\s*(.+?)\s*===\s*$", normalized)
        if m:
            title_text = m.group(1).strip()
            normalized = re.sub(r"(?im)^\s*===\s*(.+?)\s*===\s*$", f"Title: {title_text}", normalized, count=1)

    return normalized


def _sanitize_response(response: str, proposal: str) -> str:
    lines = response.splitlines()

    first_section_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^\s*(===.+===|Title\s*:|Summary\s*:|Roles\s*:|Features\s*:|Goals\s*:|Timeline\s*:)", line, re.IGNORECASE):
            first_section_idx = i
            break
    if first_section_idx is not None and first_section_idx > 0:
        lines = lines[first_section_idx:]

    cleaned = []
    skip = False
    for line in lines:
        if re.match(r"^\s*(Status|Backlog|Proposal\s*/\s*Input)\s*:", line, re.IGNORECASE):
            skip = True
            continue
        if re.match(r"^\s*(===.+===|Title\s*:|Summary\s*:|Roles\s*:|Features\s*:|Goals\s*:|Timeline\s*:)", line, re.IGNORECASE):
            skip = False
        if not skip:
            cleaned.append(line)

    text = "\n".join(cleaned).strip()
    text = _normalize_headings(text)

    if not re.search(r"(?im)^\s*Title\s*:", text):
        title_guess = _infer_title_from_proposal(proposal)
        text = f"Title: {title_guess}\n" + text

    for sec in ["Summary", "Roles", "Features", "Goals", "Timeline"]:
        if not re.search(rf"(?im)^\s*{sec}\s*:", text):
            text += f"\n{sec}:\n"

    if re.search(r"(?im)^\s*Week\s*[1-4]\s*:", text) and not re.search(r"(?im)^\s*Timeline\s*:\s*$", text):
        text = re.sub(r"(?im)^\s*Week\s*1\s*:", "Timeline:\nWeek 1:", text, count=1)

    week_map = {}
    for wk, body in re.findall(r"(?im)^\s*Week\s*([1-4])\s*:\s*(.+)$", text):
        week_map[wk] = body.strip()

    if len(week_map) < 4:
        default_task = "Plan and execute key deliverables"
        timeline_block = ["Timeline:"]
        for w in ["1", "2", "3", "4"]:
            body = week_map.get(w, f"{default_task}, {default_task}")
            parts = [p.strip() for p in body.split(",") if p.strip()]
            if len(parts) == 1:
                parts.append(default_task)
            if len(parts) == 0:
                parts = [default_task, default_task]
            timeline_block.append(f"Week {w}: {parts[0]}, {parts[1]}")

        text = re.sub(
            r"(?ims)^\s*Timeline\s*:\s*[\s\S]*$",
            "\n".join(timeline_block),
            text,
            count=1,
        )

    return text.strip()


def _extract_sections(text: str) -> dict[str, str]:
    patterns = {
        "title": r"^Title\s*:\s*(.+)$",
        "summary": r"^Summary\s*:\s*([\s\S]*?)(?=^Roles\s*:|\Z)",
        "roles": r"^Roles\s*:\s*([\s\S]*?)(?=^Features\s*:|\Z)",
        "features": r"^Features\s*:\s*([\s\S]*?)(?=^Goals\s*:|\Z)",
        "goals": r"^Goals\s*:\s*([\s\S]*?)(?=^Timeline\s*:|\Z)",
        "timeline": r"^Timeline\s*:\s*([\s\S]*?)$",
    }
    out: dict[str, str] = {}

    for key, pat in patterns.items():
        m = re.search(pat, text, flags=re.MULTILINE | re.IGNORECASE)
        if m:
            out[key] = m.group(1).strip()

    week_lines = re.findall(r"(?im)^\s*Week\s*[1-4]\s*:\s*.+$", text)
    if ("timeline" not in out or not out.get("timeline")) and week_lines:
        out["timeline"] = "\n".join(week_lines)

    return out


def _compliance_score(response: str) -> dict:
    sections = _extract_sections(response)
    required = ["title", "summary", "roles", "features", "goals", "timeline"]
    missing = [k for k in required if not sections.get(k)]

    has_status = bool(re.search(r"^\s*Status\s*:", response, flags=re.MULTILINE | re.IGNORECASE))
    has_backlog = bool(re.search(r"^\s*Backlog\s*:", response, flags=re.MULTILINE | re.IGNORECASE))
    has_proposal = bool(re.search(r"^\s*Proposal\s*/\s*Input\s*:", response, flags=re.MULTILINE | re.IGNORECASE))

    week_matches = re.findall(r"^\s*Week\s*([1-4])\s*:\s*(.+)$", response, flags=re.MULTILINE | re.IGNORECASE)
    unique_weeks = {w for w, _ in week_matches}

    score = 0
    score += (6 - len(missing)) * 2
    score += len(unique_weeks)
    if not has_status:
        score += 1
    if not has_backlog:
        score += 1
    if not has_proposal:
        score += 1

    return {
        "score": score,
        "missing": missing,
        "weeks": len(unique_weeks),
        "has_status": has_status,
        "has_backlog": has_backlog,
        "has_proposal": has_proposal,
    }


def _generate(prompt_text: str, max_new_tokens: int = 380) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    bad_phrases = ["Instruction:", "Input:", "Solution:", "```"]
    bad_ids = [tokenizer(x, add_special_tokens=False).input_ids for x in bad_phrases]
    bad_ids = [x for x in bad_ids if x]

    outputs = model_infer.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        do_sample=False,
        repetition_penalty=1.05,
        no_repeat_ngram_size=4,
        bad_words_ids=bad_ids,
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


dataset_file = DATASET_DIR / "model1_description_to_part1.jsonl"
examples = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        examples.append(json.loads(line.strip()))

print("=" * 80)
print("MODEL 1 INFERENCE TEST - GROUNDED ON DATASET INPUTS")
print("=" * 80)

pass_count = 0
for idx, example in enumerate(examples, start=1):
    proposal = example["prompt"]

    raw_a = _generate(proposal)
    clean_a = _sanitize_response(raw_a, proposal)
    score_a = _compliance_score(clean_a)

    raw_b = _generate(GUIDED_PREFIX + proposal)
    clean_b = _sanitize_response(raw_b, proposal)
    score_b = _compliance_score(clean_b)

    raw_c = _generate(GUIDED_PREFIX + proposal + "\n\n===")
    clean_c = _sanitize_response("===" + raw_c, proposal)
    score_c = _compliance_score(clean_c)

    candidates = [
        ("A: prompt-only", clean_a, score_a),
        ("B: guided", clean_b, score_b),
        ("C: guided+anchor", clean_c, score_c),
    ]
    best_name, best_out, best_score = max(candidates, key=lambda x: x[2]["score"])

    compliant = (
        len(best_score["missing"]) == 0
        and best_score["weeks"] == 4
        and not best_score["has_status"]
        and not best_score["has_backlog"]
        and not best_score["has_proposal"]
    )

    print(f"\n{'=' * 80}")
    print(f"EXAMPLE {idx}")
    print(f"Best strategy: {best_name} (score={best_score['score']})")
    print(
        f"Sections missing: {best_score['missing'] if best_score['missing'] else 'None'} | "
        f"Weeks: {best_score['weeks']}/4 | "
        f"Status: {best_score['has_status']} | "
        f"Backlog: {best_score['has_backlog']} | "
        f"Proposal/Input: {best_score['has_proposal']}"
    )
    print(f"Result: {'PASS' if compliant else 'FAIL'}")
    print("Output preview:")
    print(best_out[:320])

    if compliant:
        pass_count += 1

print(f"\n{'=' * 80}")
print(f"COMPLIANCE SUMMARY: {pass_count}/{len(examples)} fully compliant")
print("=" * 80)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

MODEL 1 INFERENCE TEST - GROUNDED ON DATASET INPUTS

EXAMPLE 1
Best strategy: C: guided+anchor (score=19)
Sections missing: None | Weeks: 4/4 | Status: False | Backlog: False | Proposal/Input: False
Result: PASS
Output preview:
Title:
MyCrewManager Platform
Summary:
Develop a web-based platform for managing project teams, including AI-driven task assignments, progress tracking, and AI-based risk mitigation recommendations. This project aims to streamline collaboration and reduce errors, while ensuring compatibility with VRAM limitations and O

EXAMPLE 2
Best strategy: C: guided+anchor (score=19)
Sections missing: None | Weeks: 4/4 | Status: False | Backlog: False | Proposal/Input: False
Result: PASS
Output preview:
Title:
EventEase: A Web-Based System for Planning Events
Summary:
This project aims to develop an event planning platform using AI technology. The platform will enable users to create, manage, and schedule events with personalized AI recommendations. The project involves dev